# X-PIE Curation — Interactive Notebook

**High-Confidence Crosslink Curation** (`x-pie-curation.py`)

This notebook runs the X-PIE *curation* module end-to-end: it filters pLink XL-MS
results by FDR, resolves protein–protein interactions (PPIs), and (optionally)
annotates them against STRING and RCSB PDB.

> Source & license: MIT  ·  DOI: 10.5281/zenodo.20523994  ·  README: `README-CURATION`

### Workflow (7 steps)
1. Load pLink result CSV(s)
2. Filter inter-protein crosslink PSMs by FDR
3. Resolve PPIs and XL site pairs
4. Apply site-pair count threshold
5. Write curated XL-MS summary files
6. Evaluate PPIs against STRING
7. Search homologous PDB structures for unreported PPIs

Use the interactive controls below to set parameters and run the pipeline.

In [1]:
# Install dependencies (pandas, numpy, requests, biopython, ipywidgets)
import sys
!{sys.executable} -m pip install -q pandas numpy requests biopython "ipywidgets>=7.5"
print('dependencies OK')

dependencies OK


In [2]:
import os
if not os.path.exists('x-pie-curation.py'):
    print('x-pie-curation.py not found in the current working directory.')
    print('Choose one:')
    print('  A) Upload the whole x-pie folder to /content and run:  %cd /content/x-pie')
    print('  B) Clone your repository, then cd into it, e.g.:')
    print('       !git clone https://github.com/<your-org>/x-pie.git')
    print('       %cd x-pie')
else:
    print('Found x-pie-curation.py in:', os.getcwd())
    print('input dir exists :', os.path.isdir('./input'))
    print('output dir exists:', os.path.isdir('./output'))

Found x-pie-curation.py in: /home/pc/x-pie_notebook
input dir exists : True
output dir exists: True


## Interactive controls

Adjust the parameters and click **Run curation**. By default the run uses
`--network-failure-mode local-only`, i.e. it performs XL-MS filtering only and
**skips** the STRING/PDB web annotation (fast, no internet needed). Tick the
checkbox to enable STRING/PDB annotation (requires internet; can be slow for
large result files such as the bundled 49 MB example).

In [3]:
import subprocess, sys, os, ipywidgets as widgets
from IPython.display import display

# --- widget styling: wide label column so no description gets truncated ---
LBL  = {'description_width': '190px'}      # label column width
ROW  = widgets.Layout(width='560px')       # total control width
WIDE = widgets.Layout(width='auto')        # let checkbox label size itself
BTN  = widgets.Layout(width='170px')

fdr_w = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1,
                            description='FDR (%)', style=LBL, layout=ROW,
                            readout_format='.1f')
msp_w = widgets.IntSlider(value=1, min=1, max=10, step=1,
                          description='min site-pairs', style=LBL, layout=ROW)
str_w = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05,
                            description='STRING score', style=LBL, layout=ROW,
                            readout_format='.2f')
idd_w = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0,
                            description='PDB identity (%)', style=LBL, layout=ROW,
                            readout_format='.0f')
ann_w = widgets.Checkbox(value=False, indent=False,
                         description='STRING / PDB annotation (needs internet)',
                         style={'description_width': 'initial'}, layout=WIDE)
run_btn = widgets.Button(description='Run curation', button_style='success', layout=BTN)
out = widgets.Output()

def run_curation(b):
    with out:
        out.clear_output()
        cmd = [sys.executable, 'x-pie-curation.py', '--non-interactive',
               '--input-dir', './input', '--output-dir', './output',
               '--fdr', str(fdr_w.value), '--min-site-pairs', str(msp_w.value),
               '--string-score-threshold', str(str_w.value),
               '--identity-threshold', str(idd_w.value)]
        if not ann_w.value:
            cmd += ['--network-failure-mode', 'local-only']
        print('>> ' + ' '.join(cmd))
        res = subprocess.run(cmd, capture_output=True, text=True)
        print(res.stdout)
        if res.returncode != 0:
            print('STDERR:\n' + res.stderr[-3000:])
        print('\n----- produced output files -----')
        for fn in ['PPI.dat','PPI_XL_Sites.dat','Reported_PPI.dat',
                   'Putative_PPI.dat','PDB_homology_results.dat']:
            p = os.path.join('./output', fn)
            if os.path.exists(p):
                print(f'  [ok] {fn}')
            else:
                print(f'  [--] {fn}  (not produced this run)')

display(widgets.VBox([fdr_w, msp_w, str_w, idd_w, ann_w, run_btn]), out)
run_btn.on_click(run_curation)

Output()

## Understanding the outputs

After a run, the following files appear in `./output`:

| File | Description |
|------|-------------|
| `PPI.dat` | Curated protein pairs with site-pair counts |
| `PPI_XL_Sites.dat` | Crosslinked residue pairs for each PPI |
| `Reported_PPI.dat` | PPIs with STRING database support |
| `Putative_PPI.dat` | Novel PPIs below threshold or absent from STRING |
| `PDB_homology_results.dat` | Homologous PDB entries for putative PPIs |

The cell below re-displays the produced files.

In [4]:
import os
for fn in ['PPI.dat','PPI_XL_Sites.dat','Reported_PPI.dat',
           'Putative_PPI.dat','PDB_homology_results.dat']:
    p = os.path.join('./output', fn)
    if os.path.exists(p):
        print(f'\n===== {fn} =====')
        with open(p) as f:
            print(f.read()[:4000])
    else:
        print(f'\n[{fn}] not produced in this run (e.g. STRING/PDB steps were skipped).')


===== PPI.dat =====
Protein1 Protein2 SitePairCount
Q71DI3 Q99880 10
P05787 Q04695 9
P62805 Q71DI3 7
P05783 P05787 5
Q71DI3 Q99878 4
P00338 P07195 3
P12956 P13010 3
P15880 P46781 3
P23528 P60709 3
P35232 Q99623 3
P36578 Q02543 3
P46779 P62910 3
P62805 Q99880 3


===== PPI_XL_Sites.dat =====
Protein1 Site1 Protein2 Site2
P00338 14 P07195 156
P00338 14 P07195 279
P00338 224 P07195 7
P05783 111 P05787 464
P05783 118 P05787 130
P05783 241 P05787 122
P05783 407 P05787 122
P05783 407 P05787 472
P05787 101 Q04695 262
P05787 101 Q04695 399
P05787 101 Q04695 400
P05787 101 Q04695 419
P05787 122 Q04695 399
P05787 295 Q04695 15
P05787 295 Q04695 297
P05787 325 Q04695 15
P05787 347 Q04695 15
P12956 189 P13010 603
P12956 287 P13010 286
P12956 287 P13010 307
P15880 173 P46781 93
P15880 176 P46781 91
P15880 176 P46781 93
P23528 92 P60709 238
P23528 127 P60709 50
P23528 144 P60709 328
P35232 186 Q99623 97
P35232 202 Q99623 262
P35232 208 Q99623 218
P36578 393 Q02543 11
P36578 399 Q02543 11
P36578 405

## Tips & troubleshooting

- **Missing columns error** → ensure your pLink CSV has: `Peptide_Type`,
  `Protein_Type`, `Score`, `Target_Decoy`, `Q-value`, `Proteins`.
- **No rows after filtering** → your FDR threshold may be too strict; raise `--fdr`.
  Make sure `Target_Decoy == 2` (target–target) rows exist.
- **STRING step fails** → check internet; verify UniProt accessions are valid.
- **PDB step slow** → queries external APIs; for many unreported PPIs expect 10+ min.
- Head-less / no-widget run:
  `python x-pie-curation.py --non-interactive --input-dir ./input --output-dir ./output --fdr 1`